# Implement Learn2Synth

## 1. Environment Setup

In [1]:
# TRAINING DURATION CONFIGURATION
# Define the total training time in minutes.
# The script will complete the current epoch when this time is reached.
TRAINING_TIME_MINUTES = (2 * 60) + 30  # 2 hours 30 minutes

# EXPERIMENT RUN NAME
# Set a custom name for this experiment run.
# All outputs (checkpoints, logs, images) will be stored in: experiments/<RUN_NAME>
RUN_NAME = "my_experiment_run"

In [2]:
# 1. Install Cornucopia (if not already present)
!pip install git+https://github.com/balbasty/cornucopia@6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b


# 2. CRITICAL FIX: Force reinstall compatible Torch, Torchvision, and Numpy
# This aligns torchvision (0.21.0) with torch (2.6.0) and fixes the NMS error.
!pip install "torch==2.6.0+cu124" "torchvision==0.21.0+cu124" "torchaudio==2.6.0+cu124" --index-url https://download.pytorch.org/whl/cu124

# 3. Ensure Numpy is version 1.x (fixes the _ARRAY_API error)
!pip install "numpy<2.0"

!pip install -U "jsonargparse[signatures]>=4.27.7"
!pip install "protobuf==3.20.3"

!pip install pandas matplotlib

  Cloning https://github.com/balbasty/cornucopia (to revision 6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b) to /tmp/pip-req-build-fv6w1caf
  Running command git clone --filter=blob:none --quiet https://github.com/balbasty/cornucopia /tmp/pip-req-build-fv6w1caf
  Running command git rev-parse -q --verify 'sha^6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b'
  Running command git fetch -q https://github.com/balbasty/cornucopia 6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b
  Running command git checkout -q 6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b
  Resolved https://github.com/balbasty/cornucopia to commit 6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for cornucopia: filename=cornucopia-0.4.0-py3-none-any.whl size=126229 sha256=32f3c0ece98d55ad8d47167f173cc8d094dbaacd7b9795e7ee3a36ce3cf0f486
  Stored in directory: /root/.cache/pip/wheels/a3/ea/27/98c72f0858dada2

## 2. Package Setup

In [3]:
# Cell 2: Setup learn2synth package
import os
import shutil

# Paths
INPUT_SOURCE = "/kaggle/input/datasets/yassienmohamed/learn2synth-sourcecode/learn2synth"
WORKING_DIR = "/kaggle/working"
PACKAGE_DIR = os.path.join(WORKING_DIR, "learn2synth")
SCRIPT_DIR = os.path.join(WORKING_DIR, "scripts")

# Create package folder
os.makedirs(PACKAGE_DIR, exist_ok=True)
os.makedirs(SCRIPT_DIR, exist_ok=True)

# Copy python files
for filename in os.listdir(INPUT_SOURCE):
    if filename.endswith(".py"):
        shutil.copyfile(os.path.join(INPUT_SOURCE, filename), os.path.join(PACKAGE_DIR, filename))

# Ensure it's a package
if not os.path.exists(os.path.join(PACKAGE_DIR, "__init__.py")):
    open(os.path.join(PACKAGE_DIR, "__init__.py"), 'a').close()

print("Library setup complete.")

Library setup complete.


## 3. Define Training Script

### 3.1 Imports and Configuration

In [4]:
%%writefile scripts/train_non_parametric_synthseg.py

import pytorch_lightning as pl
from pytorch_lightning.cli import LightningCLI
from pytorch_lightning.callbacks import ModelCheckpoint, Callback # Added Callback
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
import sys
import os
import datetime
from glob import glob
from os import path, makedirs
from ast import literal_eval
from random import shuffle
from typing import Sequence, List, Tuple, Optional, Union
import math
import fnmatch
import random

# Added for plotting
import pandas as pd 
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn.functional as F
import nibabel as nib
import cornucopia as cc
from torch.utils.data import Dataset, DataLoader
from torchmetrics.segmentation import DiceScore as dice_compute

# --- Project Imports ---
project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
sys.path.append(project_root)
sys.path.append('/kaggle/input/datasets/yassienmohamed/learn2synth-sourcecode')

from learn2synth.networks import UNet, SegNet
from learn2synth.train import SynthSeg
from learn2synth.losses import DiceLoss, LogitMSELoss, CatLoss, CatMSELoss, DiceCELoss
from learn2synth import optim
from cornucopia import SynthFromLabelTransform, LoadTransform
from learn2synth.utils import folder2files

# --- Configuration ---
DEFAULT_FOLDER = '/kaggle/input/datasets/yassienmohamed/synthseg-label-map'
input_file = 'flair.nii'
train_file = 'synthseg.nii'

Writing scripts/train_non_parametric_synthseg.py


### 3.2 Model Architecture

In [5]:
%%writefile -a scripts/train_non_parametric_synthseg.py

class Model(pl.LightningModule):
    def __init__(self,
                 ndim: int = 3,
                 nb_classes: int = 8,
                 seg_nb_levels: int = 6,
                 seg_features: Sequence[int] = (16, 32, 64, 128, 256, 512),
                 seg_activation: str = 'ReLU',
                 seg_nb_conv: int = 2,
                 seg_norm: Optional[str] = 'instance',
                 loss: str = 'dice_ce',
                 alpha: float = 1.0,
                 # Parameters for the "Real" logging branch (optional visualization)
                 real_sigma_min: float = 0.15,
                 real_sigma_max: float = 0.15,
                 real_low: float = 0.5,
                 real_middle: float = 0.5,
                 real_high: float = 0.5,
                 classic: bool = True,
                 optimizer: str = 'Adam',
                 optimizer_options: dict = dict(lr=1e-4),
                 time_limit_minutes: float = None,
                 ):
        super().__init__()
        self.save_hyperparameters()

        self.optimizer = optimizer
        self.optimizer_options = dict(optimizer_options or {})
        self.alpha = alpha
        self.time_limit_minutes = time_limit_minutes
        
        # Real image augmentation params (for validation visualization only)
        self.real_sigma_min = real_sigma_min
        self.real_sigma_max = real_sigma_max
        self.real_low = real_low
        self.real_middle = real_middle
        self.real_high = real_high
        self.classic = classic

        # --- 1. Segmentation Network (The Student) ---
        segnet = UNet(
            ndim,
            features=seg_features,
            activation=seg_activation,
            nb_levels=seg_nb_levels,
            nb_conv=seg_nb_conv,
            norm=seg_norm,
        )
        segnet = SegNet(ndim, 1, nb_classes, backbone=segnet, activation=None)

        # --- 2. Target Labels ---
        
        # References:
        # - https://surfer.nmr.mgh.harvard.edu/fswiki/SynthSeg
        # - https://surfer.nmr.mgh.harvard.edu/fswiki/WMH-SynthSeg

        # Synthesis labels (forces symmetric structures to share same GMM intensity distribution)
        synth_labels = [
            (99,),                 # Lesion
            (24, 4, 43, 5, 44, 14, 15),  # CSF + ventricles
            (3, 42, 8, 47),          # GM (cortex + cerebellar cortex)
            (2, 41, 7, 46),          # WM (cerebrum + cerebellar WM)
            (10, 49, 11, 50, 12, 51, 13, 52, 26, 58, 28, 60),  # Deep gray
            (17, 53, 18, 54),        # Limbic MTL
            (16,),                 # Brainstem
        ]

        # Target labels mapping (the final output maps the synthesis groupings back to grouped target output)
        target_labels = synth_labels


        # --- 3. Synthesis Generator (The Teacher) ---
        synth = cc.SynthFromLabelTransform(
            # A. Label Handling
            one_hot=False,          # Return integer map [0, 1]
            
            # B. Geometric Deformations (Crucial for Cortical Thickening)
            elastic=0.05,           # Strong elastic deformation to change thickness
            elastic_nodes=10,       # Wavy deformations
            rotation=15,            # +/- 15 degrees
            shears=0.012,
            zooms=0.15,

            # C. MRI Simulation (Crucial for FCD Subtlety)
            resolution=5,           # [0-5x] downsampling. Simulates partial volume / blurring at GW junction.
            motion_fwhm=2.0,        # [0-2mm] blur. Simulates "fuzzy" borders characteristic of FCD.
            snr=10,                 # Low SNR. Hides the lesion in noise so strict intensity isn't enough.
            gmm_fwhm=10,            # High texture smoothing. Makes lesion look like a cohesive tissue block.
            gamma=0.5,              # Aggressive contrast scrambling. Forces learning shape over brightness.
            
            # D. Artifacts
            bias=7,                 # Smooth bias field
            bias_strength=0.5,      # Strong intensity inhomogeneity
        )

        # Wrap in SharedSynth to ensure Real validation images get same geometric warp
        synth = cc.batch(SharedSynth(synth))

        # --- 4. Loss Function ---
        if loss == 'dice':
            loss = DiceLoss(activation='Softmax')
        elif loss == 'logitmse':
            loss = LogitMSELoss()
        elif loss == 'cat':
            loss = CatLoss(activation='Softmax')
        elif loss == 'catmse':
            loss = CatMSELoss(activation='Softmax')
        elif loss == 'dice_ce':
            loss = DiceCELoss(
                weighted=False,        # Critical: Keep False or use mild fixed weights
                lambda_dice=1.0,       # Primary driver for overlap
                lambda_ce=1.0,         # Stabilizer for gradients
                activation='Softmax'
                )
        elif loss == 'focal_tversky':
            loss = FocalTverskyLoss(activation='Softmax')
        else:
            raise ValueError('Unsupported loss', loss)

        self.network = SynthSeg(segnet, synth, loss)

        # Manual optimization control (required for Learn2Synth/SynthSeg framework)
        self.automatic_optimization = False
        self.network.set_backward(self.manual_backward)
        
        # Initialize Dice metric once for validation
        self.val_dice = dice_compute(average='micro', include_background=False, num_classes=len(target_labels) + 1, input_format="index")
        self.val_fcd_dice = dice_compute(average='none', include_background=False, num_classes=len(target_labels) + 1, input_format="index")
        # Placeholder for any batch outputs if needed
        self._val_batches = []

    

    def configure_optimizers(self):
        optimizer = getattr(optim, self.optimizer)
        optimizer_init = lambda x: optimizer(x, **(self.optimizer_options or {}))
        optimizers = self.network.configure_optimizers(optimizer_init)
        self.network.set_optimizers(self.optimizers)
        return optimizers

    def training_step(self, batch, batch_idx):
        if self.trainer.current_epoch % 10 == 0 and batch_idx == 0:
            torch.cuda.empty_cache()

        # loss_real is calculated but detached (no gradient)
        loss_synth, loss_real = self.network.synth_and_train_step(*batch)
        
        # Combined for logging purposes only
        loss = loss_synth + self.alpha * loss_real
        
        self.log(f'train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss_synth, loss_real, pred_synth, pred_real, \
            synth_image, synth_ref, real_image, real_ref \
            = self.network.synth_and_eval_for_plot(*batch)

        pred_labels = pred_real.argmax(dim=1)
        target_labels = real_ref.squeeze(1)

        # Update Dice metric (accumulation)
        self.val_dice.update(pred_labels.cpu(), target_labels.cpu())
        self.val_fcd_dice.update(pred_labels.cpu(), target_labels.cpu())

        # --- FIX 3: Robust Image Saving ---
        # Only save on the first batch of the validation set
        if batch_idx == 0:
            epoch = self.trainer.current_epoch
            
            # Construct a stable path. Use the trainer's log_dir which we fixed in the CLI.
            # Fallback to default_root_dir if log_dir is not yet set.
            base_dir = self.trainer.log_dir or self.trainer.default_root_dir
            image_root = os.path.join(base_dir, 'images')
            
            makedirs(image_root, exist_ok=True)

            pred_synth_argmax = pred_synth[0].argmax(dim=0)
            pred_real_argmax = pred_labels[0]

            # Log class counts for debugging
            self.log('pred_synth_num_classes', float(len(torch.unique(pred_synth_argmax))), prog_bar=False)
            self.log('pred_real_num_classes', float(len(torch.unique(pred_real_argmax))), prog_bar=False)

            # Save explicitly every 10 epochs
            if epoch % 10 == 0:
                print(f"\n[Saving] Sample images for Epoch {epoch} to {image_root}...") # Console confirmation
                save(pred_synth_argmax, f'{image_root}/epoch-{epoch:04d}_synth-pred.nii.gz')
                save(pred_real_argmax, f'{image_root}/epoch-{epoch:04d}_real-pred.nii.gz')
                save(synth_image[0].squeeze(0), f'{image_root}/epoch-{epoch:04d}_synth-image.nii.gz')
                save(real_image[0].squeeze(0), f'{image_root}/epoch-{epoch:04d}_real-image.nii.gz')
                save(synth_ref[0].squeeze(0).to(torch.uint8), f'{image_root}/epoch-{epoch:04d}_synth-ref.nii.gz')
                save(real_ref[0].squeeze(0).to(torch.uint8), f'{image_root}/epoch-{epoch:04d}_real-ref.nii.gz')

        # Log loss
        loss = loss_synth + self.alpha * loss_real
        self.log('eval_loss', loss, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        # Compute Dice over all validation batches
        dice_epoch = self.val_dice.compute()
        fcd_dice_epoch = self.val_fcd_dice.compute()[-1]
        
        # --- FIX 1: Console Output Sanitization ---
        # 1. Log to system (Tensorboard/CSV)
        self.log('val_dice', dice_epoch, prog_bar=True)
        self.log('fcd_dice', fcd_dice_epoch, prog_bar=True)
        
        # 2. Force print to console (New Line) so it isn't eaten by the progress bar
        # We access the current training loss if available, otherwise N/A
        train_loss = self.trainer.callback_metrics.get('train_loss', -1)
        eval_loss = self.trainer.callback_metrics.get('eval_loss', -1)
        
        print(f"\n" + "="*40)
        print(f"EPOCH {self.trainer.current_epoch} SUMMARY:")
        print(f"  Train Loss : {train_loss:.4f}")
        print(f"  Eval Loss  : {eval_loss:.4f}")
        print(f"  DICE SCORE : {dice_epoch:.4f}") # <--- Critical Metric Visible
        print(f"  FCD DICE   : {fcd_dice_epoch:.4f}")
        print(f"="*40 + "\n")
        
        # Reset metric for next epoch
        self.val_dice.reset()
        self.val_fcd_dice.reset()

    def configure_callbacks(self):
        callbacks = []
        if self.time_limit_minutes is not None:
            callbacks.append(TimeLimitCallback(self.time_limit_minutes))
        return callbacks

    def forward(self, x):
        return self.network(x)

Appending to scripts/train_non_parametric_synthseg.py


### 3.3 Helper Functions

In [6]:
%%writefile -a scripts/train_non_parametric_synthseg.py

def save(dat, fname):
    dat = dat.detach().cpu().numpy()
    h = nib.Nifti1Header()
    h.set_data_dtype(dat.dtype)
    nib.save(nib.Nifti1Image(dat, np.eye(4), h), fname)

class TimeLimitCallback(Callback):
    def __init__(self, limit_minutes):
        self.limit_minutes = limit_minutes
        self.start_time = None

    def on_train_start(self, trainer, pl_module):
        env_limit = os.environ.get('TRAINING_TIME_MINUTES')
        if env_limit is not None:
            try:
                self.limit_minutes = float(env_limit)
            except ValueError:
                pass
        self.start_time = datetime.datetime.now()
        print(f"\n[TimeLimit] Training started at {self.start_time}. Limit: {self.limit_minutes} mins.")

    def on_validation_epoch_end(self, trainer, pl_module):
        if self.limit_minutes and self.start_time:
            elapsed = datetime.datetime.now() - self.start_time
            if elapsed.total_seconds() > self.limit_minutes * 60:
                print(f"\n[TimeLimit] Time limit reached ({elapsed.total_seconds()/60:.1f} > {self.limit_minutes} mins). Stopping after this epoch.")
                trainer.should_stop = True

class LossGraphCallback(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        # --- Path Configuration ---
        # Robustly handle the log directory. We use trainer.log_dir directly
        # because we flattened the folder structure (removed 'version_X' subfolders).
        if not trainer.log_dir:
            return
            
        metrics_path = os.path.join(trainer.log_dir, "metrics.csv")
        plot_path = os.path.join(trainer.log_dir, "training_plot.png")
        
        # Abort if metrics file doesn't exist yet (e.g., first sanity check)
        if not os.path.exists(metrics_path):
            return

        # --- Data Loading & Aggregation ---
        try:
            # Read the CSV log generated by Lightning
            metrics = pd.read_csv(metrics_path)
        except Exception:
            # Handle potential race conditions (file being written to)
            return

        # The CSVLogger logs every step. We group by 'epoch' and take the mean
        # to get a clean, smooth line for plotting.
        epoch_metrics = metrics.groupby("epoch").mean()

        # --- Plotting ---
        plt.figure(figsize=(10, 6))
        
        # 1. Train Loss (Primary metric for convergence)
        if 'train_loss' in epoch_metrics:
            plt.plot(epoch_metrics.index, epoch_metrics['train_loss'], 
                     label='Train Loss', color='blue', linestyle='-', alpha=0.7)
            
        # 2. Validation Loss (Check for overfitting)
        if 'eval_loss' in epoch_metrics:
            plt.plot(epoch_metrics.index, epoch_metrics['eval_loss'], 
                     label='Val Loss', color='red', linestyle='--')
            
        # 3. Validation Dice Score (Primary metric for segmentation quality)
        # Note: We look for 'val_dice' to match the key logged in the Model class.
        if 'val_dice' in epoch_metrics:
            plt.plot(epoch_metrics.index, epoch_metrics['val_dice'], 
                     label='Val Dice', color='green', linewidth=2)

        # Formatting
        plt.title(f'Training Metrics (Epoch {trainer.current_epoch})')
        plt.xlabel('Epoch')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)

        # --- Saving ---
        # Save directly to the run root so it is easily accessible.
        try:
            plt.savefig(plot_path, dpi=150)
        finally:
            plt.close()
            

class SharedSynth(torch.nn.Module):
    """
    Wrapper that applies the exact same geometric deformation to both 
    the Synthetic branch (generation) and the Real branch (augmentation).
    """
    def __init__(self, synth):
        super().__init__()
        self.synth = synth

    def forward(self, slab, img, lab):
        # --- CRITICAL FIX: Disable 16-bit mixed precision for Cornucopia ---
        # GMMs and Quantiles fail/underflow in 16-bit. Force float32 for synthesis.
        with torch.amp.autocast('cuda', enabled=False):
            # Ensure incoming floating tensors are strictly 32-bit
            if img.is_floating_point():
                img = img.float()
                
            # 1. Sample random parameters based on the label map 'slab'
            final = self.synth.make_final(slab, 1)
            final.deform = final.deform.make_final(slab)
            
            # 2. Generate Synthetic Image
            simg, slab = final(slab)
            
            # 3. Apply SAME deformation to Real Image
            rimg, rlab = final.deform([img, lab])
            rimg = final.intensity(rimg)
            rlab = final.postproc(rlab)
            
        return simg, slab, rimg, rlab

Appending to scripts/train_non_parametric_synthseg.py


### 3.4 Data Loading

In [7]:
%%writefile -a scripts/train_non_parametric_synthseg.py

class PairedDataset(Dataset):
    def __init__(self, ndim, images, labels, split_synth_real=True,
                 subset=None, device=None):
        self.ndim = ndim
        self.device = device
        self.split_synth_real = split_synth_real
        self.labels = np.asarray(folder2files(labels)[subset or slice(None)])
        self.images = np.asarray(folder2files(images)[subset or slice(None)])
        
        assert len(self.labels) == len(self.images), \
            "Number of labels and images don't match"

    def __len__(self):
        n = len(self.images)
        if self.split_synth_real:
            n = n // 2
        return n

    def __getitem__(self, idx):
        # Load Real Label and Real Image
        lab = str(self.labels[idx])
        img = str(self.images[idx])

        lab = LoadTransform(ndim=self.ndim, dtype=torch.long, device=self.device)(lab)
        img = LoadTransform(ndim=self.ndim, dtype=torch.float32, device=self.device)(img)

        if self.split_synth_real:
            # If split, use a DIFFERENT label map for synthesis than for validation
            slab = str(self.labels[len(self) + idx])
            slab = LoadTransform(ndim=self.ndim, dtype=torch.long, device=self.device)(slab)
            return slab, img, lab
        else:
            # Shared: Use the SAME label map for synthesis and validation
            return lab, img, lab


class PairedDataModule(pl.LightningDataModule):
    def __init__(self,
                 ndim: int,
                 images: Optional[Sequence[str]] = None,
                 labels: Optional[Sequence[str]] = None,
                 eval: Union[str, slice, List[int], int, float] = 0.2,
                 test: Union[str, slice, List[int], int, float] = 0.2,
                 preshuffle: bool = True,
                 shared: bool = True,
                 batch_size: int = 64,
                 shuffle: bool = False,
                 num_workers: int = 4,
                 prefetch_factor: int = 2,
                 ):
        super().__init__()
        self.ndim = ndim

        # --- DATA LOADING ---
        if labels is None or images is None:
            # Search for sub-XXXX folders
            subject_folders = sorted(glob(path.join(DEFAULT_FOLDER, 'sub-*')))
            self.labels = []
            self.images = []

            print(f"Found {len(subject_folders)} subject folders. Scanning...")

            for subj_dir in subject_folders:
                # Pair FusedMask (Label) with FLAIR (Image)
                label_path = path.join(subj_dir, f'{train_file}')
                image_path = path.join(subj_dir, f'{input_file}')

                if path.exists(label_path) and path.exists(image_path):
                    self.labels.append(label_path)
                    self.images.append(image_path)
            
            print(f"Successfully loaded {len(self.labels)} pairs.")
        else:
            self.labels = list(labels)
            self.images = list(images)

        assert len(self.images) == len(self.labels), "Mismatch in file counts!"

        self.eval = parse_eval(eval)
        self.test = parse_eval(test)
        self.preshuffle = preshuffle
        self.shared = shared
        self.train_kwargs = dict(batch_size=batch_size, shuffle=shuffle, 
                               num_workers=num_workers, prefetch_factor=prefetch_factor)
        self.eval_kwargs = dict(batch_size=batch_size, shuffle=False, 
                               num_workers=num_workers, prefetch_factor=prefetch_factor)

    def setup(self, stage):
        images, labels = self.images, self.labels
        
        if self.preshuffle:
            combined = list(zip(images, labels))
            shuffle(combined)
            images, labels = zip(*combined)
            images, labels = list(images), list(labels)

        # Calculate split indices
        def get_count(param, total):
            if isinstance(param, float): return int(math.ceil(total * param))
            return 0 # Default fallback

        n_eval = get_count(self.eval, len(images))
        n_test = get_count(self.test, len(images))

        # Split: Test -> Eval -> Train
        self.test_images, self.test_labels = images[:n_test], labels[:n_test]
        remaining_images = images[n_test:]
        remaining_labels = labels[n_test:]
        
        self.eval_images, self.eval_labels = remaining_images[:n_eval], remaining_labels[:n_eval]
        self.train_images, self.train_labels = remaining_images[n_eval:], remaining_labels[n_eval:]

    def train_dataloader(self):
        return DataLoader(PairedDataset(self.ndim, self.train_images, self.train_labels, 
                                      split_synth_real=not self.shared), **self.train_kwargs)

    def val_dataloader(self):
        return DataLoader(PairedDataset(self.ndim, self.eval_images, self.eval_labels, 
                                      split_synth_real=not self.shared), **self.eval_kwargs)

    def test_dataloader(self):
        return DataLoader(PairedDataset(self.ndim, self.test_images, self.test_labels, 
                                      split_synth_real=not self.shared), **self.eval_kwargs)


Appending to scripts/train_non_parametric_synthseg.py


### 3.5 Main Execution

In [8]:
%%writefile -a scripts/train_non_parametric_synthseg.py

def parse_eval(eval):
    if not isinstance(eval, str): return eval
    try: return literal_eval(eval)
    except: return eval

class CLI(LightningCLI):
    def add_arguments_to_parser(self, parser):
        parser.add_lightning_class_args(ModelCheckpoint, "checkpoint")
        parser.set_defaults({
            "checkpoint.monitor": "eval_loss",
            "checkpoint.save_last": True,
            "checkpoint.save_top_k": 5,
            # Ensure checkpoints are saved inside our organized structure
            "checkpoint.filename": "checkpoint-{epoch:02d}-{eval_loss:.2f}-{val_dice:.2f}",
        })

    def instantiate_trainer(self, **kwargs):
        # --- FIX 2: Output Directory Architecture ---
        # 1. Determine Run Name (Environment Variable or Timestamp)
        if os.environ.get("L2S_RUN_NAME"):
            run_name = os.environ.get("L2S_RUN_NAME")
        else:
            run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            run_name = f"run_{run_timestamp}"
        
        # 2. Get the base root (default: lightning_logs)
        default_root = kwargs.get("default_root_dir", "experiments")
        
        # 3. Create the unified path: experiments/run_20230101_120000/
        save_dir = os.path.join(default_root, run_name)
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"[System] Initializing Run: {run_name}")
        print(f"[System] All artifacts will be stored in: {save_dir}")

        # 4. Configure Loggers to point to this EXACT directory
        # We set version='' to prevent Lightning from creating 'version_0' subfolders
        logger = [
            TensorBoardLogger(save_dir=default_root, name=run_name, version=''), 
            CSVLogger(save_dir=default_root, name=run_name, version='')
        ]
        
        # 5. Add Callbacks
        callbacks = kwargs.get("callbacks", [])
        if callbacks is None: callbacks = []
        callbacks.append(LossGraphCallback())
        
        # 6. Update Trainer kwargs
        kwargs["default_root_dir"] = save_dir # Update root so checkpoints inherit it
        kwargs["logger"] = logger
        kwargs["callbacks"] = callbacks
        
        return super().instantiate_trainer(**kwargs)


if __name__ == '__main__':
    cli = CLI(Model, PairedDataModule)

Appending to scripts/train_non_parametric_synthseg.py


## 4. Run Training

In [9]:
!L2S_RUN_NAME={RUN_NAME} TRAINING_TIME_MINUTES={TRAINING_TIME_MINUTES} python /kaggle/working/scripts/train_non_parametric_synthseg.py fit \
    --data.ndim 3 \
    --model.ndim 3 \
    --model.nb_classes 8 \
    --data.batch_size 1 \
    --data.num_workers 4 \
    --data.shared true \
    --trainer.max_epochs 1000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 16-mixed \
    --model.seg_nb_levels 5 \
    --model.seg_features "[16, 32, 64, 128, 256]" \
    --trainer.enable_progress_bar false \
    --trainer.log_every_n_steps 10 \
    --checkpoint.save_top_k 3 \
    --checkpoint.monitor eval_loss \
    --checkpoint.mode min \
    --model.time_limit_minutes {TRAINING_TIME_MINUTES}

/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/seed.py:44: No seed found, seed set to 0
Seed set to 0
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)
Found 78 subject folders. Scanning...
Successfully loaded 78 pairs.
[System] Initializing Run: my_experiment_run
[System] All artifacts will be stored in: experiments/my_experiment_run
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-02-19 14:17:31.268099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771510651.617007     171 cuda_dnn.cc:8579]

In [10]:
# import shutil
# import os

# exp_dir = "/kaggle/working/experiments"

# if os.path.exists(exp_dir):
#     shutil.rmtree(exp_dir)